<a href="https://colab.research.google.com/github/GustavoNachbar/churn-dataset-clusters-classify-tests/blob/main/xgboost_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd

df = pd.read_csv("/content/Churn_Modelling.csv")
# Renomeia as colunas do inglês original para português
df = df.rename(columns={
    'RowNumber': 'NumeroLinha',
    'CustomerId': 'IdCliente',
    'Surname': 'Sobrenome',
    'CreditScore': 'PontuacaoCredito',
    'Geography': 'Geografia',
    'Gender': 'Genero',
    'Age': 'Idade',
    'Tenure': 'TempoRelacionamento',
    'Balance': 'Saldo',
    'NumOfProducts': 'NumeroProdutos',
    'HasCrCard': 'PossuiCartaoCredito',
    'IsActiveMember': 'MembroAtivo',
    'EstimatedSalary': 'SalarioEstimado',
    'Exited': 'Saiu'
})

variaveis_numericas = [
    'PontuacaoCredito', 'Idade', 'TempoRelacionamento',
    'Saldo', 'NumeroProdutos', 'SalarioEstimado'
]

# 70% treino, 30% temporário
df_treino, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["Saiu"]
)

# 15% teste, 15% validação
df_teste, df_validacao = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["Saiu"]
)

# Cluster treinado SÓ com cancelados do df_treino (nunca olha teste/validação)
df_cancelados_treino = df_treino[df_treino["Saiu"] == 1][variaveis_numericas]

scaler_perfil = StandardScaler()
X_scaled_cancelados = scaler_perfil.fit_transform(df_cancelados_treino)

kmeans_k4 = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_k4.fit(X_scaled_cancelados)

# Aplicado (predict) nas 3 bases separadamente, sem re-treinar em nenhuma delas
for dataframe in [df_treino, df_teste, df_validacao]:
    X_scaled = scaler_perfil.transform(dataframe[variaveis_numericas])
    dataframe['Cluster_k4'] = kmeans_k4.predict(X_scaled)
    dataframe['Distancia_Centroide_k4'] = kmeans_k4.transform(X_scaled).min(axis=1)

In [6]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef

features = variaveis_numericas + ['Cluster_k4', 'Distancia_Centroide_k4']

X_train = df_treino[features]
y_train = df_treino['Saiu']
X_test = df_teste[features]
y_test = df_teste['Saiu']

param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search_xgb = GridSearchCV(
    estimator=XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1),
    param_grid=param_grid_xgb, cv=5, scoring='f1', n_jobs=-1
)
grid_search_xgb.fit(X_train, y_train)

cv_resultados = pd.DataFrame(grid_search_xgb.cv_results_)

top10 = cv_resultados.sort_values('mean_test_score', ascending=False).head(10)[
    ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
]
top10['algoritmo'] = 'XGBoost'  # trocar em cada notebook: Random_Forest, XGBoost, SVM, Naive_Bayes

top10.to_csv('top10_boost.csv', index=False)  # trocar o nome do arquivo em cada notebook

melhor_modelo = grid_search_xgb.best_estimator_
y_pred = melhor_modelo.predict(X_test)
y_proba = melhor_modelo.predict_proba(X_test)[:, 1]

resultado_xgb_k4 = {
    'melhores_parametros': grid_search_xgb.best_params_,
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_proba),
     'mcc': matthews_corrcoef(y_test, y_pred)
}
pd.DataFrame([resultado_xgb_k4])

,melhores_parametros,accuracy,precision,recall,f1,roc_auc,mcc
0,"{'colsample_bytree': 1.0, 'learning_rate': 0.1...",0.85,0.736842,0.411765,0.528302,0.840373,0.47431
